# Notebook 09C — Missingness Mechanism and Logistic Model Stabilization

**Goal:** Explain why all 2024 unknown-label filings lack key financial/governance fields, check for source/join artifacts, and test whether the strongest Notebook 09B logistic candidate converges reliably.

**Scope:** Read-only source diagnostics and **2021–2022 validation-only** model comparison. No 2023–2024 label-based tuning, no new investigation queue, and no changes to Notebooks 08, 09, 09A or 09B.

**Interpretation:** `fraud_label=1` means **self-reported significant diversion of assets**, not proven fraud. Unknown labels are never treated as negatives. Risk scores are not calibrated fraud probabilities.

## 1. Configuration and source discovery

Run from the project root or its `notebooks` directory. Notebook 09B manifest supplies the feature names and split. Uses discovered NCCS Parquet filenames rather than hard-coded patterns. The source checks only need 2024 files.

In [ ]:
from pathlib import Path
import json, gc, time, warnings
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display

SEED = 42
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
INPUT = ROOT / "data/processed/nccs_longitudinal_features_v1.parquet"
RAW_DIR = ROOT / "data/external/nccs/efile_v3_1"
OUT = ROOT / "data/processed/notebook09c"
OUT.mkdir(parents=True, exist_ok=True)
B_DIR = ROOT / "data/processed/notebook09b"
N9_DIR = ROOT / "data/processed/notebook09"
manifest_path = N9_DIR / "model_manifest.json"
for p in (INPUT, manifest_path, RAW_DIR):
    assert p.exists(), f"Required input missing: {p}"
manifest = json.loads(manifest_path.read_text())
FEATURES = list(manifest["features"])
assert manifest["train_years"] == [2010, 2020]
assert manifest["validation_years"] == [2021, 2022]
assert "fraud_label" not in FEATURES

def sqlpath(p):
    return str(p).replace("'", "''")

con = duckdb.connect()
con.execute("SET threads = 4")
con.execute("SET memory_limit = '6GB'")
con.execute(f"CREATE VIEW features AS SELECT * FROM read_parquet('{sqlpath(INPUT)}')")
feature_columns = set(con.execute("DESCRIBE features").df().column_name)
assert set(FEATURES + ["OBJECTID", "ein", "tax_year", "fraud_label"]) <= feature_columns

summary_2024 = sorted(p for p in RAW_DIR.rglob("*.parquet") if "SUMMARY" in p.name.upper() and "2024" in p.name)
governance_2024 = sorted(p for p in RAW_DIR.rglob("*.parquet") if "GOVERNANCE" in p.name.upper() and "2024" in p.name)
header_2024 = sorted(p for p in RAW_DIR.rglob("*.parquet") if "HEADER" in p.name.upper() and "2024" in p.name)
print("Project:", ROOT)
print("Output:", OUT)
print("2024 SUMMARY/GOVERNANCE/HEADER files:", len(summary_2024), len(governance_2024), len(header_2024))
print("Feature count:", len(FEATURES))
print("2024 feature rows:", con.execute("SELECT COUNT(*) FROM features WHERE tax_year=2024").fetchone()[0])

## 2. Characterize the known-versus-unknown split

Measure label availability, missingness patterns, and the relationship between label status and feature availability. This checks whether unknown-label rows form a separate source population; it does **not** assume the reason.

In [ ]:
q = con.execute("""
SELECT
 CASE WHEN fraud_label IN (0,1) THEN 'known' ELSE 'unknown' END AS label_status,
 COUNT(*) AS filings,
 COUNT(*) FILTER (WHERE total_assets IS NULL) AS missing_assets,
 COUNT(*) FILTER (WHERE total_liabilities IS NULL) AS missing_liabilities,
 COUNT(*) FILTER (WHERE voting_members IS NULL) AS missing_voting,
 COUNT(*) FILTER (WHERE independent_members IS NULL) AS missing_independent,
 COUNT(*) FILTER (WHERE total_revenue IS NULL) AS missing_revenue,
 COUNT(*) FILTER (WHERE total_expenses IS NULL) AS missing_expenses,
 COUNT(*) FILTER (WHERE total_assets IS NULL AND voting_members IS NULL) AS missing_both,
 COUNT(*) FILTER (WHERE total_assets IS NOT NULL AND voting_members IS NOT NULL) AS complete_both
FROM features WHERE tax_year=2024 GROUP BY 1 ORDER BY 1
""").df()
for f in ["assets", "liabilities", "voting", "independent", "revenue", "expenses", "both"]:
    q[f"missing_{f}_pct"] = 100 * q[f"missing_{f}"] / q["filings"]
display(q)
q.to_csv(OUT / "01_2024_label_and_missingness_structure.csv", index=False)

cross = con.execute("""
SELECT CASE WHEN fraud_label IN (0,1) THEN 'known' ELSE 'unknown' END AS label_status,
       CASE WHEN total_assets IS NULL THEN 'missing' ELSE 'present' END AS assets,
       CASE WHEN voting_members IS NULL THEN 'missing' ELSE 'present' END AS governance,
       COUNT(*) AS filings
FROM features WHERE tax_year=2024 GROUP BY 1,2,3 ORDER BY 1,2,3
""").df()
display(cross)
cross.to_csv(OUT / "02_2024_missingness_cross_tab.csv", index=False)

## 3. Verify 2024 raw source coverage and join keys

We distinguish **no raw source row**, **raw source NULL**, and **a populated raw source value**. Use `OBJECTID`, not EIN, to match a specific return. The raw files may contain multiple filings for the same EIN/year; only the selected final `OBJECTID` is matched. Inspect schemas before making assumptions.

In [ ]:
if not (summary_2024 and governance_2024):
    print("2024 raw source files not found. Skip raw-source comparisons and inspect RAW_DIR.")
else:
    summary_paths = [str(p) for p in summary_2024]
    governance_paths = [str(p) for p in governance_2024]
    con.execute(f"CREATE VIEW summary24 AS SELECT * FROM read_parquet({repr(summary_paths)}, union_by_name=true)")
    con.execute(f"CREATE VIEW governance24 AS SELECT * FROM read_parquet({repr(governance_paths)}, union_by_name=true)")
    for name in ("summary24", "governance24"):
        schema = con.execute(f"DESCRIBE {name}").df()
        print(name, "rows:", con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0])
        print(name, "duplicate OBJECTIDs:", con.execute(f"SELECT COUNT(*) - COUNT(DISTINCT OBJECTID) FROM {name}").fetchone()[0])
        display(schema.head(12))
    assert con.execute("SELECT COUNT(*) - COUNT(DISTINCT OBJECTID) FROM summary24").fetchone()[0] == 0, "Duplicate raw SUMMARY OBJECTIDs: inspect before joining"
    assert con.execute("SELECT COUNT(*) - COUNT(DISTINCT OBJECTID) FROM governance24").fetchone()[0] == 0, "Duplicate raw GOVERNANCE OBJECTIDs: inspect before joining"
    raw_coverage = con.execute("""
      SELECT CASE WHEN f.fraud_label IN (0,1) THEN 'known' ELSE 'unknown' END AS label_status,
             COUNT(*) AS filings,
             COUNT(*) FILTER (WHERE s.OBJECTID IS NULL) AS no_summary_record,
             COUNT(*) FILTER (WHERE g.OBJECTID IS NULL) AS no_governance_record,
             COUNT(*) FILTER (WHERE s.OBJECTID IS NOT NULL) AS summary_record_present,
             COUNT(*) FILTER (WHERE g.OBJECTID IS NOT NULL) AS governance_record_present
      FROM features f
      LEFT JOIN summary24 s ON f.OBJECTID=s.OBJECTID
      LEFT JOIN governance24 g ON f.OBJECTID=g.OBJECTID
      WHERE f.tax_year=2024
      GROUP BY 1 ORDER BY 1
    """).df()
    display(raw_coverage)
    raw_coverage.to_csv(OUT / "03_2024_raw_source_coverage.csv", index=False)

## 4. Raw-field values versus feature NULLs

Check if source data actually lacks the fields or if the pipeline loses them. This uses the six mappings previously verified in Notebook 09A. A zero discrepancy count is expected; unexpected results should halt interpretation.

In [ ]:
raw_mappings = {
    "total_assets": ("summary24", "F9_01_NAFB_ASSET_TOT_EOY"),
    "total_liabilities": ("summary24", "F9_01_NAFB_LIAB_TOT_EOY"),
    "total_revenue": ("summary24", "F9_01_REV_TOT_CY"),
    "total_expenses": ("summary24", "F9_01_EXP_TOT_CY"),
    "voting_members": ("governance24", "F9_06_GVRN_NUM_VOTING_MEMB"),
    "independent_members": ("governance24", "F9_06_GVRN_NUM_VOTING_MEMB_IND"),
}
raw_rows=[]
if summary_2024 and governance_2024:
    for field, (view, raw_col) in raw_mappings.items():
        available = set(con.execute(f"DESCRIBE {view}").df().column_name)
        if raw_col not in available:
            print("Missing raw column:", view, raw_col)
            continue
        data = con.execute(f"""
          SELECT CASE WHEN f.fraud_label IN (0,1) THEN 'known' ELSE 'unknown' END AS label_status,
            COUNT(*) AS filings,
            COUNT(*) FILTER (WHERE r.OBJECTID IS NULL) AS missing_raw_record,
            COUNT(*) FILTER (WHERE r.OBJECTID IS NOT NULL AND r."{raw_col}" IS NULL) AS raw_null,
            COUNT(*) FILTER (WHERE r."{raw_col}" IS NOT NULL) AS raw_populated,
            COUNT(*) FILTER (WHERE r."{raw_col}" IS NOT NULL AND f."{field}" IS NULL) AS raw_present_feature_null,
            COUNT(*) FILTER (WHERE r."{raw_col}" IS NULL AND f."{field}" IS NOT NULL) AS raw_null_feature_present
          FROM features f LEFT JOIN {view} r USING (OBJECTID)
          WHERE f.tax_year=2024 GROUP BY 1
        """).df()
        data.insert(0,"feature",field)
        raw_rows.append(data)
    if raw_rows:
        raw_detail = pd.concat(raw_rows,ignore_index=True)
        display(raw_detail)
        raw_detail.to_csv(OUT / "04_2024_raw_field_vs_final_feature.csv",index=False)
        print("Raw populated -> final NULL:", int(raw_detail.raw_present_feature_null.sum()))
        print("Raw NULL -> final populated:", int(raw_detail.raw_null_feature_present.sum()))

## 5. Label-availability mechanism by year

Determine whether the 2024 pattern is longstanding. The grouped query avoids loading millions of rows into pandas. This is descriptive, not a causal explanation.

In [ ]:
years = con.execute("""
SELECT tax_year,
       CASE WHEN fraud_label IN (0,1) THEN 'known' ELSE 'unknown' END AS label_status,
       COUNT(*) AS filings,
       ROUND(100.0 * COUNT(*) FILTER (WHERE total_assets IS NULL)/COUNT(*),3) AS missing_assets_pct,
       ROUND(100.0 * COUNT(*) FILTER (WHERE voting_members IS NULL)/COUNT(*),3) AS missing_voting_pct,
       ROUND(100.0 * COUNT(*) FILTER (WHERE total_revenue IS NULL)/COUNT(*),3) AS missing_revenue_pct
FROM features GROUP BY 1,2 ORDER BY 1,2
""").df()
display(years)
years.to_csv(OUT / "05_label_missingness_by_tax_year.csv", index=False)

## 6. Model stabilization setup

Recreate the Notebook 09B winning candidate: 18 features, **all positive training examples** plus sampled negatives, 1 million training filings. Compare `liblinear` at higher iteration limits against `lbfgs` with scaling. Capture convergence warnings and iteration counts. Selection is **validation-only**; this is further exploration on a repeatedly used validation period, not an unbiased estimate.

Set `RUN_MODEL_FITS = False` if you only want the data audit. A million rows can require substantial RAM and time.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.exceptions import ConvergenceWarning

RUN_MODEL_FITS = True
TRAIN_CAP = 1_000_000

# Keep this aligned with Notebook 09B; no holdout labels are loaded.
select_cols = ["tax_year", "ein", "OBJECTID", "fraud_label"] + FEATURES
quoted = ", ".join('"' + c.replace('"', '""') + '"' for c in select_cols)

def load_dev(lo, hi):
    d = con.execute(f"""
        SELECT {quoted} FROM features
        WHERE tax_year BETWEEN ? AND ? AND fraud_label IN (0,1)
        ORDER BY tax_year, ein, OBJECTID
    """, [lo,hi]).df()
    for f in FEATURES:
        d[f] = pd.to_numeric(d[f], errors="coerce").replace([np.inf,-np.inf],np.nan).astype("float32")
    assert not d.duplicated(["ein","tax_year"]).any()
    return d

def sample_train(df, cap=TRAIN_CAP):
    pos=df.loc[df.fraud_label.eq(1)]
    neg=df.loc[df.fraud_label.eq(0)]
    take=min(len(neg), cap-len(pos))
    assert take>=0
    neg=neg.sample(n=take,random_state=SEED) if take<len(neg) else neg
    return pd.concat([pos,neg],ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)

def metrics(y,score):
    y=np.asarray(y,dtype=np.int8)
    score=np.asarray(score,dtype=float)
    assert np.isfinite(score).all()
    order=np.argsort(-score,kind="stable")
    out={"n":len(y),"positives":int(y.sum()),"prevalence":float(y.mean()),
         "average_precision":float(average_precision_score(y,score)),
         "roc_auc":float(roc_auc_score(y,score)) if len(np.unique(y))==2 else np.nan}
    for k in (100,250,500,1000):
        out[f"found_at_{k}"]=int(y[order[:k]].sum())
    return out

if RUN_MODEL_FITS:
    train=load_dev(2010,2020)
    val=load_dev(2021,2022)
    fit=sample_train(train)
    yval=val.fraud_label.astype("int8").to_numpy()
    print("Train sample:",len(fit),"positives:",int(fit.fraud_label.sum()))
    print("Validation:",len(val),"positives:",int(yval.sum()))
    print("Validation prevalence:",round(float(yval.mean()),7))

## 7. Fit stabilized logistic candidates

Warnings and `n_iter_` are recorded rather than silently ignored. If a solver still warns, treat its ranking metrics as provisional. No test labels or queue are used here.

In [ ]:
CANDIDATES = [
    ("09b_liblinear_400", "liblinear", "robust", 400),
    ("liblinear_1500", "liblinear", "robust", 1500),
    ("lbfgs_1500", "lbfgs", "standard", 1500),
]

results=[]
validation_predictions={}
if RUN_MODEL_FITS:
    for name,solver,scaling,max_iter in CANDIDATES:
        print("Fitting:",name,flush=True)
        scaler=RobustScaler() if scaling=="robust" else StandardScaler()
        model=Pipeline([
            ("impute",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),
            ("scale",scaler),
            ("model",LogisticRegression(C=0.1,class_weight="balanced",solver=solver,
                                       max_iter=max_iter,random_state=SEED)),
        ])
        start=time.monotonic()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always",ConvergenceWarning)
            model.fit(fit[FEATURES],fit.fraud_label.astype("int8"))
        conv=[str(w.message) for w in caught if issubclass(w.category,ConvergenceWarning)]
        score=model.predict_proba(val[FEATURES])[:,1]
        validation_predictions[name]=score.astype("float32")
        row={"candidate":name,"solver":solver,"scaler":scaling,"max_iter":max_iter,
             "n_iter_used":int(np.max(model.named_steps["model"].n_iter_)),
             "convergence_warnings":len(conv),"warning_text":" | ".join(conv),
             "seconds":round(time.monotonic()-start,1),**metrics(yval,score)}
        results.append(row)
        pd.DataFrame(results).to_csv(OUT/"06_logistic_stabilization_validation.csv",index=False)
        print("AP:",round(row["average_precision"],6),"found@500:",row["found_at_500"],
              "iterations:",row["n_iter_used"],"warnings:",len(conv),flush=True)
        del model,score
        gc.collect()
    result_df=pd.DataFrame(results).sort_values("average_precision",ascending=False)
    display(result_df[["candidate","average_precision","roc_auc","found_at_100","found_at_500","found_at_1000","n_iter_used","convergence_warnings","seconds"]])

## 8. Year-specific validation stability and comparison to Notebook 09B

Assess whether the ranking improvement persists in both 2021 and 2022. Use this to understand robustness, not to cherry-pick the best test result.

In [ ]:
if RUN_MODEL_FITS:
    year_rows=[]
    for name,score in validation_predictions.items():
        for yr in (2021,2022):
            mask=val.tax_year.to_numpy()==yr
            year_rows.append({"candidate":name,"tax_year":yr,**metrics(yval[mask],score[mask])})
    year_df=pd.DataFrame(year_rows)
    display(year_df[["candidate","tax_year","n","positives","average_precision","roc_auc","found_at_100","found_at_500","found_at_1000"]])
    year_df.to_csv(OUT/"07_logistic_validation_by_year.csv",index=False)

    b_comparison=B_DIR/"01_validation_experiment_comparison.csv"
    if b_comparison.exists():
        prev=pd.read_csv(b_comparison)
        print("Notebook 09B top validation experiments:")
        display(prev.sort_values("average_precision",ascending=False).head(5)[["experiment","average_precision","roc_auc","found_at_500","found_at_1000"]])
    else:
        print("09B comparison CSV not found:",b_comparison)

## 9. Decision summary and guardrails

Outputs are diagnostics. **Do not replace** `notebook09/selected_model.joblib` or the existing 2024 queue based on these exploratory results. A separate population-scoring strategy may be required if unknown-label rows have fundamentally different predictor availability. Model validation should eventually use a genuinely untouched later period or external data, with a documented point-in-time cutoff.

In [ ]:
summary={
    "source_feature_file":str(INPUT),
    "raw_2024_summary_files":len(summary_2024),
    "raw_2024_governance_files":len(governance_2024),
    "target":"Self-reported significant diversion of assets, not verified fraud",
    "training_years":[2010,2020],"validation_years":[2021,2022],
    "holdout_years_not_used_for_selection":[2023,2024],
    "run_model_fits":RUN_MODEL_FITS,
    "model_results":results,
    "scores_are_calibrated_probabilities":False,
    "existing_model_and_queue_unchanged":True,
    "limitations":[
        "Unknown labels cannot be evaluated as positives or negatives",
        "Labeled versus unlabeled feature availability may differ structurally",
        "Repeated use of 2021–2022 validation can overfit selection",
        "2023–2024 holdout was inspected in earlier notebooks and is not pristine",
        "No point-in-time filing availability audit has been completed",
    ],
}
(OUT/"08_09c_diagnostic_manifest.json").write_text(json.dumps(summary,indent=2,default=str))
print("Saved diagnostic outputs to:",OUT)
if RUN_MODEL_FITS and results:
    stable=[r for r in results if r["convergence_warnings"]==0]
    if stable:
        best=max(stable,key=lambda r:r["average_precision"])
        print("Best warning-free candidate by validation AP:",best["candidate"],
              "AP:",round(best["average_precision"],6),"top 500:",best["found_at_500"])
    else:
        print("No warning-free candidate. Do not treat any candidate as stabilized.")
print("Review source coverage, missingness, convergence, and year-by-year stability before Notebook 10.")